Your First GPU Program!
========================

This example demonstrates basic GPU computing using Python.
We'll use Numba CUDA - it works even without a GPU (falls back to CPU).

If running on Google Colab:
1. Go to Runtime > Change runtime type
2. Select "GPU" as Hardware accelerator
3. Run this code!

## GPU Programming Model

### Key Concepts

1. **Kernel**: A function that runs on the GPU
2. **Thread**: Smallest unit of execution
3. **Block**: Group of threads (up to 1024)
4. **Grid**: Collection of blocks


```
Grid
├── Block (0,0)
│   ├── Thread (0,0)
│   ├── Thread (0,1)
│   └── Thread (0,2)
├── Block (0,1)
│   ├── Thread (0,0)
│   ├── Thread (0,1)
│   └── Thread (0,2)
└── Block (1,0)
    ├── Thread (0,0)
    ├── Thread (0,1)
    └── Thread (0,2)
```

In [7]:
import numpy as np
try:
    from numba import cuda
    CUDA_AVAILABLE = cuda.is_available()
except ImportError:
    print("⚠️  Numba not installed. Install with: pip install numba")
    CUDA_AVAILABLE = False

print(f"🖥️  CUDA Available: {CUDA_AVAILABLE}")


🖥️  CUDA Available: True


# ============================================================================
# EXAMPLE 1: Hello World from GPU
# ============================================================================


In [8]:
@cuda.jit
def hello_gpu(result):
    """
    A simple GPU kernel that runs on each thread.
    Each thread writes its ID to the result array.
    """
    # Get the thread's unique position
    thread_id = cuda.grid(1)  # 1D grid

    if thread_id < result.shape[0]:
        result[thread_id] = thread_id

def example1_hello():
    """
    Creates an array of 10 zeros, sends it to GPU, and
    has each GPU thread write its own ID.

    BEFORE GPU:
    Array: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

    GPU LAUNCHES 32 threads (in 1 block):
    Thread 0  → writes 0 to result[0]
    Thread 1  → writes 1 to result[1]
    Thread 2  → writes 2 to result[2]
    ...
    Thread 9  → writes 9 to result[9]
    Thread 10-31 → do nothing (idx >= 10 check prevents them)
    """

    print("\n" + "="*60)
    print("EXAMPLE 1: Hello GPU!")
    print("="*60)

    # Create array on CPU (host)
    n = 10
    result = np.zeros(n, dtype=np.int32)

    print(f"Array before GPU: {result}")

    if CUDA_AVAILABLE:
        # Define grid and block dimensions
        threads_per_block = 32
        blocks = (n + threads_per_block - 1) // threads_per_block

        print(f"Launching kernel with {blocks} blocks, {threads_per_block} threads per block")

        # Launch kernel
        hello_gpu[blocks, threads_per_block](result)

        print(f"Array after GPU:  {result}")
        print("Each thread wrote its ID!")
    else:
        print("⚠️  No GPU available, skipping GPU execution")

# ============================================================================
# EXAMPLE 2: Vector Addition (The "Hello World" of GPU Computing)
# ============================================================================


In [9]:
# The @cuda.jit decorator is Numba's Just-In-Time (JIT)
# compiler that converts your Python function
# into a GPU kernel (CUDA code

@cuda.jit
def vector_add_gpu(a, b, result):
    """
    Add two vectors element-wise on the GPU.
    Each thread computes __one__ element: result[i] = a[i] + b[i]
    If we have 1 million elements, you launch 1 million threads!
    All additions happen simultaneously on the GPU.
    """
    idx = cuda.grid(1)

    if idx < result.shape[0]:
        result[idx] = a[idx] + b[idx]

def vector_add_cpu(a, b):
    """CPU version for comparison"""
    return a + b # NumPy does this sequentially (or with few cores)


def example2_vector_addition():
    print("\n" + "="*60)
    print("EXAMPLE 2: Vector Addition")
    print("="*60)

    # Create test data
    n = 1_000_000
    a = np.random.rand(n).astype(np.float32)
    b = np.random.rand(n).astype(np.float32)

    # CPU version
    import time
    start = time.time()
    result_cpu = vector_add_cpu(a, b)
    cpu_time = time.time() - start
    print(f"⏱️  CPU time: {cpu_time*1000:.2f} ms")

    if CUDA_AVAILABLE:
        # GPU version
        result_gpu = np.zeros(n, dtype=np.float32)

        threads_per_block = 256
        blocks = (n + threads_per_block - 1) // threads_per_block

        # Warm up GPU
        # # First call: Compiles Python → GPU code (slow!)
        vector_add_gpu[blocks, threads_per_block](a, b, result_gpu)
        cuda.synchronize()  # WAIT for GPU to finish!

        # Timed run
        start = time.time()
        # # Second call: Uses cached compiled version (fast!)
        vector_add_gpu[blocks, threads_per_block](a, b, result_gpu)
        cuda.synchronize()  # Wait for GPU to finish
        gpu_time = time.time() - start

        print(f"⏱️ GPU time: {gpu_time*1000:.2f} ms")
        print(f"🚀 Speedup: {cpu_time/gpu_time:.2f}x")

        # Verify correctness
        matches = np.allclose(result_cpu, result_gpu)
        print(f"✅ Results match: {matches}")

        if not matches:
            print(f" Max difference: {np.max(np.abs(result_cpu - result_gpu))}")
    else:
        print("⚠️  No GPU available for comparison")




# ============================================================================
# EXAMPLE 3: Understanding Thread Indexing
# ============================================================================


In [10]:

@cuda.jit
def show_thread_info(result):
    """
    Each thread reports its position in the grid.
    Understanding this is crucial for CUDA programming!

    This demonstrates how GPU threads are organized.
    Visually:
GRID (entire GPU)
│
├─ BLOCK 0 (32 threads)
│  ├─ Thread 0:  threadIdx.x=0,  blockIdx.x=0  →  global ID = 0*32 + 0  = 0
│  ├─ Thread 1:  threadIdx.x=1,  blockIdx.x=0  →  global ID = 0*32 + 1  = 1
│  ├─ Thread 2:  threadIdx.x=2,  blockIdx.x=0  →  global ID = 0*32 + 2  = 2
│  └─ ...
│  └─ Thread 31: threadIdx.x=31, blockIdx.x=0  →  global ID = 0*32 + 31 = 31
│
├─ BLOCK 1 (32 threads)
│  ├─ Thread 0:  threadIdx.x=0,  blockIdx.x=1  →  global ID = 1*32 + 0  = 32
│  ├─ Thread 1:  threadIdx.x=1,  blockIdx.x=1  →  global ID = 1*32 + 1  = 33
│  ├─ Thread 2:  threadIdx.x=2,  blockIdx.x=1  →  global ID = 1*32 + 2  = 34
│  └─ ...
│  └─ Thread 31: threadIdx.x=31, blockIdx.x=1  →  global ID = 1*32 + 31 = 63
│
├─ BLOCK 2 (32 threads)
│  ├─ Thread 0:  threadIdx.x=0,  blockIdx.x=2  →  global ID = 2*32 + 0  = 64
│  └─ ...
│  └─ Thread 31: threadIdx.x=31, blockIdx.x=2  →  global ID = 2*32 + 31 = 95
│
└─ BLOCK 3 (32 threads) - only first 4 do work!
   ├─ Thread 0:  threadIdx.x=0,  blockIdx.x=3  →  global ID = 3*32 + 0  = 96
   ├─ Thread 1:  threadIdx.x=1,  blockIdx.x=3  →  global ID = 3*32 + 1  = 97
   ├─ Thread 2:  threadIdx.x=2,  blockIdx.x=3  →  global ID = 3*32 + 2  = 98
   ├─ Thread 3:  threadIdx.x=3,  blockIdx.x=3  →  global ID = 3*32 + 3  = 99
   └─ Thread 4-31: IDLE (idx >= 100, so the if-check prevents them from working)

    """
    # Thread position within its block
    tx = cuda.threadIdx.x

    # Block position in the grid
    bx = cuda.blockIdx.x

    # Block size
    bd = cuda.blockDim.x

    # Global thread ID (unique across all threads)
    idx = cuda.grid(1)  # This is equivalent to: bx * bd + tx

    if idx < result.shape[0]:
        result[idx] = idx

def example3_thread_indexing():
    """
    # nice video on thread indexing: https://www.youtube.com/watch?v=cRY5utouJzQ
    # Calculation:
    # Block 0: threads 0-31
    # Block 1: threads 32-63
    # Block 2: threads 64-95
    # Block 3: threads 96-127 (but only 96-99 do work!)
    """
    print("\n" + "="*60)
    print("EXAMPLE 3: Understanding Thread Indexing")
    print("="*60)

    if not CUDA_AVAILABLE:
        print("⚠️  GPU required for this example")
        return

    n = 100
    result = np.zeros(n, dtype=np.int32)

    # Use 32 threads per block
    threads_per_block = 32
    blocks = (n + threads_per_block - 1) // threads_per_block



    print(f"Total threads needed: {n}")
    print(f"Threads per block: {threads_per_block}")
    print(f"Number of blocks: {blocks}")
    print(f"Total threads launched: {blocks * threads_per_block}")
    print(f"Extra threads (idle): {blocks * threads_per_block - n}")

    show_thread_info[blocks, threads_per_block](result)

    print(f"\nFirst 10 thread IDs: {result[:10]}")
    print(f"Last 10 thread IDs: {result[-10:]}")
    print("\n💡 Key insight: Each thread gets a unique ID from 0 to N-1")



# ============================================================================
# MAIN PROGRAM
# ============================================================================


In [11]:
def main():
    print("🎓 Welcome to GPU Programming!")
    print("=" * 60)

    if CUDA_AVAILABLE:
        # Print GPU info
        print(f"GPU Device: {cuda.get_current_device().name.decode()}")
        print(f"Compute Capability: {cuda.get_current_device().compute_capability}")

    # Run examples
    example1_hello()
    example2_vector_addition()
    example3_thread_indexing()

    print("\n" + "="*60)
    print("Congratulations! You've run your first GPU programs!")
    print("="*60)
    print("\n💡 Key Takeaways:")
    print("   1. Each GPU thread has a unique ID")
    print("   2. Threads are organized into blocks")
    print("   3. Many threads run in parallel")
    print("   4. GPU shines with large data (100K+ elements)")
    print("\n👉 Next: Learn about memory management!")

if __name__ == "__main__":
    main()


🎓 Welcome to GPU Programming!
GPU Device: Tesla T4
Compute Capability: (7, 5)

EXAMPLE 1: Hello GPU!
Array before GPU: [0 0 0 0 0 0 0 0 0 0]
Launching kernel with 1 blocks, 32 threads per block
Array after GPU:  [0 1 2 3 4 5 6 7 8 9]
Each thread wrote its ID!

EXAMPLE 2: Vector Addition
⏱️  CPU time: 0.89 ms
⏱️ GPU time: 6.89 ms
🚀 Speedup: 0.13x
✅ Results match: True

EXAMPLE 3: Understanding Thread Indexing
Total threads needed: 100
Threads per block: 32
Number of blocks: 4
Total threads launched: 128
Extra threads (idle): 28


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:697: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))
/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:937: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))
/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:937: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))
/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:697: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))



First 10 thread IDs: [0 1 2 3 4 5 6 7 8 9]
Last 10 thread IDs: [90 91 92 93 94 95 96 97 98 99]

💡 Key insight: Each thread gets a unique ID from 0 to N-1

Congratulations! You've run your first GPU programs!

💡 Key Takeaways:
   1. Each GPU thread has a unique ID
   2. Threads are organized into blocks
   3. Many threads run in parallel
   4. GPU shines with large data (100K+ elements)

👉 Next: Learn about memory management!


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:937: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))
